# Week 12: เอเจนต์ AI และการใช้เครื่องมือ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aofphy/SCI193611_ARTIFICIAL_INTELLIGENCE/blob/main/labs/w12_agent.ipynb)

**Objective:** เขียนลูปเอเจนต์เอง **โดยไม่ใช้เฟรมเวิร์ก** เพื่อให้เห็นว่าทุกอย่างเกิดขึ้นที่ไหน

1. เครื่องมือและ registry
2. ชั้นควบคุม 5 ข้อ
3. ลูปของเอเจนต์
4. โมเดลจำลองสำหรับทดสอบแบบออฟไลน์
5. ชุดทดสอบและการวัดผล

ส่วนที่ 1 ถึง 5 รันได้ทันที ส่วนที่ 6 ต่อกับโมเดลจริง

## 1) เครื่องมือ

เครื่องมือที่ดีสำหรับโมเดล ต่างจาก API ที่ดีสำหรับโปรแกรมเมอร์:
ชื่อเป็นกริยาชัดเจน คำอธิบายบอกว่า**เมื่อไรควรใช้** และข้อความผิดพลาดบอกวิธีแก้

In [ ]:
import json, math, re, pathlib

CONSTANTS = {"c": 2.99792458e8, "h": 6.62607015e-34, "e": 1.602176634e-19,
             "k_B": 1.380649e-23, "N_A": 6.02214076e23, "G": 6.67430e-11}

def lookup_constant(name: str) -> str:
    """ค้นค่าคงที่ทางฟิสิกส์ในหน่วย SI ใช้เมื่อผู้ใช้ถามถึงค่าคงที่ เช่น c, h, e"""
    if name not in CONSTANTS:
        return f"ไม่มีค่าคงที่ชื่อ {name!r} ที่มีให้คือ {sorted(CONSTANTS)}"
    return f"{name} = {CONSTANTS[name]:.9g} (SI)"

def calculate(expression: str) -> str:
    """คำนวณนิพจน์คณิตศาสตร์ ใช้เมื่อต้องการผลลัพธ์ตัวเลขที่แม่นยำ"""
    if not re.fullmatch(r"[0-9eE+\-*/(). ,a-z_]+", expression):
        return "นิพจน์มีอักขระที่ไม่อนุญาต ใช้ได้เฉพาะตัวเลขและตัวดำเนินการ"
    try:
        env = {"__builtins__": {}, **{k: getattr(math, k) for k in
               ("sqrt", "pi", "e", "log", "exp", "sin", "cos")}}
        return str(eval(expression, env))     # ponytail: env ถูกจำกัดไว้แล้ว
    except Exception as ex:
        return f"คำนวณไม่ได้: {ex}"

def count_lines(path: str) -> str:
    """นับจำนวนบรรทัดของไฟล์ ใช้เมื่อผู้ใช้ถามขนาดหรือความยาวของไฟล์"""
    p = pathlib.Path(path)
    if not p.is_file():
        return f"ไม่พบไฟล์ {path!r} ให้ระบุพาธที่มีอยู่จริง"
    return f"{path}: {len(p.read_text(errors='ignore').splitlines())} บรรทัด"

REGISTRY = {f.__name__: f for f in (lookup_constant, calculate, count_lines)}
READ_ONLY = set(REGISTRY)          # แล็บนี้ยังไม่มีเครื่องมือที่เขียนอะไร

def tool_schemas():
    """สร้าง JSON Schema จาก signature และ docstring (แบบเดียวกับที่ MCP ทำให้)"""
    import inspect
    out = []
    for name, fn in REGISTRY.items():
        params = {p: {"type": "string"} for p in inspect.signature(fn).parameters}
        out.append({"type": "function", "function": {
            "name": name, "description": inspect.getdoc(fn),
            "parameters": {"type": "object", "properties": params,
                           "required": list(params)}}})
    return out

print(json.dumps(tool_schemas()[0], ensure_ascii=False, indent=1))

## 2) ชั้นควบคุม

**นี่คือจุดที่ควบคุมทุกอย่าง** โมเดลไม่ได้รันอะไรเลย มันแค่ *ขอ*
โค้ดของเราเป็นคนตัดสินใจว่าจะรันหรือไม่

In [ ]:
MAX_RESULT = 2000

def execute(name, args, approve=None):
    """รันเครื่องมือหนึ่งตัวพร้อมการควบคุม 5 ชั้น คืนสตริงเสมอ ไม่โยน exception"""
    if name not in REGISTRY:                                        # 1 มีจริงไหม
        return f"ไม่มีเครื่องมือชื่อ {name!r} ที่มีคือ {sorted(REGISTRY)}"

    import inspect
    want = set(inspect.signature(REGISTRY[name]).parameters)
    if set(args) != want:                                           # 2 อาร์กิวเมนต์ถูกไหม
        return f"อาร์กิวเมนต์ผิด {name} ต้องการ {sorted(want)} แต่ได้ {sorted(args)}"

    if name not in READ_ONLY and approve and not approve(name, args):  # 3 ต้องอนุมัติไหม
        return "ผู้ใช้ปฏิเสธการดำเนินการนี้"

    try:
        return str(REGISTRY[name](**args))[:MAX_RESULT]              # 4 จำกัดขนาดผล
    except Exception as ex:                                          # 5 ส่ง error กลับ
        return f"เครื่องมือทำงานผิดพลาด: {type(ex).__name__}: {ex}"

assert "ไม่มีเครื่องมือ" in execute("send_email", {"to": "x"})
assert "อาร์กิวเมนต์ผิด" in execute("calculate", {"expr": "1+1"})
assert execute("calculate", {"expression": "2+2"}) == "4"
assert "ไม่มีค่าคงที่" in execute("lookup_constant", {"name": "zzz"})
assert "ไม่พบไฟล์" in execute("count_lines", {"path": "/no/such/file"})
print("OK: ชั้นควบคุมจับทุกกรณี และไม่มีกรณีไหนที่โปรแกรมพัง")

## 3) ลูปของเอเจนต์

เงื่อนไขหยุดสามชั้น: **จำนวนขั้น**, **โมเดลตอบแล้ว**, และ **งบประมาณ**
เอเจนต์ที่ไม่มีสามอย่างนี้คือบั๊กที่รอเกิด ไม่ใช่คุณสมบัติ

In [ ]:
SYSTEM = ("คุณเป็นผู้ช่วยด้านฟิสิกส์ ใช้เครื่องมือเมื่อจำเป็น "
          "อย่าเดาค่าตัวเลข ให้ค้นจากเครื่องมือเสมอ")

def run_agent(task, llm, max_steps=8, budget=0.50, trace=None):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    spent = 0.0

    for step in range(max_steps):
        reply, cost = llm(messages, tool_schemas())
        spent += cost
        messages.append(reply)
        if trace is not None:
            trace.append({"step": step, "spent": round(spent, 4), **reply})

        if not reply.get("tool_calls"):
            return reply.get("content", ""), spent, step + 1
        if spent > budget:
            return "หยุดเพราะเกินงบประมาณ", spent, step + 1

        for call in reply["tool_calls"]:
            result = execute(call["name"], call["args"])
            messages.append({"role": "tool", "name": call["name"], "content": result})
            if trace is not None:
                trace.append({"step": step, "tool": call["name"],
                              "args": call["args"], "result": result[:120]})

    return "หยุดเพราะครบจำนวนขั้นสูงสุด", spent, max_steps

## 4) โมเดลจำลอง

`ScriptedLLM` ทำตัวเหมือนโมเดลจริงพอที่จะทดสอบลูปได้ รวมถึง**พฤติกรรมที่ผิดพลาด**
เช่น เรียกเครื่องมือที่ไม่มีจริง เพื่อพิสูจน์ว่าชั้นควบคุมของเราทำงาน

In [ ]:
class ScriptedLLM:
    """โมเดลจำลองที่ตอบตามกฎ ใช้ทดสอบลูปโดยไม่ต้องมี API"""
    COST = 0.002

    def __call__(self, messages, tools):
        seen = {m.get("name") for m in messages if m.get("role") == "tool"}
        task = messages[1]["content"]

        if "ค่าคงที่" in task and "lookup_constant" not in seen:
            name = "h" if "พลังค์" in task else "c"
            return {"role": "assistant", "tool_calls":
                    [{"name": "lookup_constant", "args": {"name": name}}]}, self.COST

        if "พลังงาน" in task and "calculate" not in seen:
            return {"role": "assistant", "tool_calls":
                    [{"name": "calculate",
                      "args": {"expression": "6.62607015e-34 * 5e14"}}]}, self.COST

        if "บรรทัด" in task and "count_lines" not in seen:
            return {"role": "assistant", "tool_calls":
                    [{"name": "count_lines", "args": {"path": "README.md"}}]}, self.COST

        if "หลอน" in task and not seen:                # จำลองการแต่งชื่อเครื่องมือ
            return {"role": "assistant", "tool_calls":
                    [{"name": "search_internet", "args": {"q": "x"}}]}, self.COST

        facts = [m["content"] for m in messages if m.get("role") == "tool"]
        return {"role": "assistant", "content": "สรุป: " + " | ".join(facts)}, self.COST

llm = ScriptedLLM()
trace = []
answer, cost, steps = run_agent("ค่าคงที่ของพลังค์มีค่าเท่าไร", llm, trace=trace)
print(answer, f"\n(cost={cost:.4f}, steps={steps})\n")
for t in trace:
    print(" ", t)

In [ ]:
# เอเจนต์ต้องกู้คืนได้เมื่อโมเดลเรียกเครื่องมือที่ไม่มีจริง
trace = []
answer, _, _ = run_agent("ทดสอบการหลอนของเครื่องมือ", llm, trace=trace)
assert any("ไม่มีเครื่องมือ" in str(t.get("result", "")) for t in trace)
print("OK: เอเจนต์ได้รับข้อความผิดพลาดกลับไป และทำงานต่อได้แทนที่จะพัง")

# เพดานงบประมาณต้องทำงาน
class Looper:
    def __call__(self, messages, tools):
        return {"role": "assistant", "tool_calls":
                [{"name": "calculate", "args": {"expression": "1+1"}}]}, 0.30

out, spent, steps = run_agent("วนไปเรื่อย ๆ", Looper(), budget=0.50)
assert "งบประมาณ" in out and steps < 8, (out, steps)

out, _, steps = run_agent("วนไปเรื่อย ๆ", Looper(), budget=99, max_steps=4)
assert "ขั้นสูงสุด" in out and steps == 4
print("OK: เงื่อนไขหยุดทั้งงบประมาณและจำนวนขั้นทำงานถูกต้อง")

## 5) ชุดทดสอบ

ต่างจากการประเมินพรอมป์ตตรงที่เราสนใจ **เส้นทาง** ไม่ใช่แค่คำตอบสุดท้าย
เขียน `check` ที่ตรวจได้ด้วยโปรแกรม เพื่อให้รันซ้ำได้ทุกครั้งที่แก้อะไร

In [ ]:
TESTS = [
    {"task": "ค่าคงที่ของพลังค์มีค่าเท่าไร",
     "check": lambda out, tr: "6.626" in out},
    {"task": "ค่าคงที่ความเร็วแสงมีค่าเท่าไร",
     "check": lambda out, tr: "2.998" in out or "2.99792" in out},
    {"task": "คำนวณพลังงานของโฟตอนที่ความถี่ 5e14 Hz",
     "check": lambda out, tr: any(t.get("tool") == "calculate" for t in tr)},
    {"task": "README.md มีกี่บรรทัด",
     "check": lambda out, tr: any(t.get("tool") == "count_lines" for t in tr)},
]

def run_suite(llm, tests=TESTS):
    rows = []
    for t in tests:
        tr = []
        out, cost, steps = run_agent(t["task"], llm, trace=tr)
        rows.append({"task": t["task"][:34], "pass": bool(t["check"](out, tr)),
                     "steps": steps, "cost": round(cost, 4)})
    return rows

rows = run_suite(llm)
print(f"{'task':36s} {'pass':6s} {'steps':6s} cost")
for r in rows:
    print(f"{r['task']:36s} {str(r['pass']):6s} {r['steps']:<6d} {r['cost']}")
print(f"\naccuracy = {sum(r['pass'] for r in rows)}/{len(rows)}")

## 6) ต่อกับโมเดลจริง

`make_real_llm` แปลงรูปแบบของ OpenAI ให้เข้ากับลูปด้านบน
สังเกตว่า **ลูปไม่ต้องแก้เลย** เพราะเราแยกส่วนติดต่อโมเดลออกมาตั้งแต่ต้น

**ทางเลือกที่ไม่เสียเงิน** สมัคร [openrouter.ai](https://openrouter.ai/) เอา key ใส่
`OPENROUTER_API_KEY` แล้วใช้โมเดลที่ลงท้ายด้วย `:free` ดูรายชื่อที่ใช้ได้ตอนนี้ด้วย
`python llm.py --free` ข้อแลกเปลี่ยนคือมีเพดานคำขอต่อนาทีและต่อวัน
และคิวอาจยาวช่วงคนใช้เยอะ

ข้อควรระวังของสัปดาห์นี้: เอเจนต์ต้อง **เรียกเครื่องมือได้** ซึ่งโมเดลฟรีบางตัวทำไม่ได้
กรองเอาเฉพาะตัวที่ทำได้ด้วย

```bash
python llm.py --free --tools
export LLM_MODEL=<ชื่อโมเดลที่เลือก>
```

และเพราะรุ่นฟรีมีเพดานคำขอต่อนาที เอเจนต์ที่วนหลายขั้นจะชน 429 ได้ง่าย
ให้ลด `MAX_STEPS` หรือเว้นจังหวะระหว่างงานเมื่อรันชุดทดสอบ


In [ ]:
try:                                  # ไคลเอนต์กลางของแล็บสัปดาห์ 8 ถึง 14
    import llm as api
except ImportError:                   # บน Colab ที่มีแต่ไฟล์สมุดบันทึก ให้ดึงมาก่อน
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/aofphy/"
        "SCI193611_ARTIFICIAL_INTELLIGENCE/main/labs/llm.py", "llm.py")
    import llm as api

PRICE = {"in": 0.15 / 1e6, "out": 0.60 / 1e6}   # แก้ตามผู้ให้บริการ รุ่น :free คิด 0


def make_real_llm(provider=None, model=None):
    """คืนฟังก์ชันหน้าตาเหมือน `llm` จำลองด้านบน แต่ยิงโมเดลจริง"""
    def real(messages, tools):
        msgs = [{k: v for k, v in m.items() if k in
                 ("role", "content", "name", "tool_calls")} for m in messages]
        m, usage = api.complete(msgs, provider=provider, model=model,
                                tools=tools, temperature=0)
        cost = (usage.get("prompt_tokens", 0) * PRICE["in"]
                + usage.get("completion_tokens", 0) * PRICE["out"])
        calls = [{"name": c["function"]["name"],
                  "args": json.loads(c["function"]["arguments"])}
                 for c in (m.get("tool_calls") or [])]
        out = {"role": "assistant", "content": m.get("content") or ""}
        if calls:
            out["tool_calls"] = calls
        return out, cost
    return real


print(api.describe(api.resolve()))

# TODO: เอาคอมเมนต์ออกแล้วรันกับโมเดลจริง
# for r in run_suite(make_real_llm()): print(r)


## TODO และการส่งงาน

**TODO**
1. เพิ่มเครื่องมืออีกอย่างน้อย 2 ตัว โดยหนึ่งในนั้นต้องเป็นเครื่องมือที่ **เขียนไฟล์**
   แล้วใส่ไว้นอก `READ_ONLY` เพื่อให้ต้องผ่านการอนุมัติ
2. เขียนฟังก์ชัน `approve` ที่ถามผู้ใช้จริง แล้วทดสอบทั้งกรณีอนุมัติและปฏิเสธ
3. เพิ่มการตรวจจับการเรียกซ้ำ: ถ้าเรียกเครื่องมือเดิมด้วยอาร์กิวเมนต์เดิมสองครั้ง ให้หยุด
4. ขยาย `TESTS` ให้ครบ 10 งาน พร้อมฟังก์ชัน `check` ที่ตรวจด้วยโปรแกรมได้
5. เปรียบเทียบโมเดลจริง 2 ตัว บนชุดทดสอบเดียวกัน

**ส่งงาน:** โค้ดเอเจนต์, ไฟล์ร่องรอยการทำงาน (`trace`) อย่างน้อย 3 งาน,
และตารางเปรียบเทียบโมเดล 2 ตัว (อัตราสำเร็จ, ต้นทุน, จำนวนขั้นเฉลี่ย)
พร้อมข้อสรุปว่าควรเลือกตัวไหนเพราะอะไร